# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a practical guide for loading and exploring the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL below.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. We also inspect general metadata.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load Croissant Dataset
dataset = mlc.Dataset(url)

# Show the main metadata fields
md = dataset.metadata
print(f"Name:    {md.name}")
print(f"Version: {md.version}")
print(f"Identifier: {md.identifier}")
print(f"Description: {md.description}")
print(f"Keywords: {md.keywords}")
print(f"Published: {md.datePublished}")
print(f"License: {md.license}")

## 2. Data Overview
Review available record sets and their fields. All references to entities use their `@id` as required by the Croissant specification.

In [ ]:
# List all record sets in the dataset
from mlcroissant._src.structure.nodes.record_set import RecordSet

record_set_ids = [rs['@id'] for rs in dataset.metadata.to_json().get('recordSet', [])]
print('Available record sets and their @id:')
for rs in dataset.metadata.to_json().get('recordSet', []):
    print(f"  @id: {rs['@id']}, name: {rs.get('name', 'N/A')}")

# For example purposes, we'll examine fields in the first record set (if present)
if record_set_ids:
    first_record_set_id = record_set_ids[0]
    # Fetch the full record set object (or its fields)
    record_sets_json = dataset.metadata.to_json()['recordSet']
    record_set_json = next(rs for rs in record_sets_json if rs['@id'] == first_record_set_id)
    field_ids = [f['@id'] for f in record_set_json.get('field', [])]
    print(f"\nFields for record set '@id': {first_record_set_id}")
    for field in record_set_json.get('field', []):
        print(f"  Field @id: {field['@id']}  name: {field.get('name', 'N/A')}")

else:
    print('No record sets found!')

## 3. Data Extraction
Load data from one or more record sets into a Pandas DataFrame for further exploration. Refer to record set and field `@id`s from the overview above.

In [ ]:
# Prepare list of record set `@id`s
record_sets = record_set_ids
dataframes = {}

for record_set in record_sets:
    # NOTE: Use @id explicitly
    print(f"Loading records for recordSet '@id': {record_set}")
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df

if record_sets:
    chosen_record_set = record_sets[0]
    print(f"\nColumns for record set '@id': {chosen_record_set}")
    print(dataframes[chosen_record_set].columns.tolist())
    display(dataframes[chosen_record_set].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by attributes. Below, we demonstrate numeric filtering and normalization, referencing all variables by their `@id`.

**Modify the field `@id`s below as appropriate for your record set.**

In [ ]:
# For demonstration, let's choose a likely-numeric field among the first few columns
# If unsure, print the first rows to inspect field names. Update the two variables accordingly.

# You may need to update these depending on dataset structure:
numeric_field_id = None
group_field_id = None

chosen_df = dataframes[chosen_record_set]

# Attempt to autodetect a numeric column
for col in chosen_df.columns:
    if chosen_df[col].dtype.kind in 'iuf':
        numeric_field_id = col
        break
# Attempt to pick a string/categorical column
for col in chosen_df.columns:
    if col != numeric_field_id and chosen_df[col].dtype == object:
        group_field_id = col
        break

print(f"Numeric field detected (by @id): {numeric_field_id}")
print(f"Categorical field detected (by @id): {group_field_id}")

# Skip EDA if not found
if numeric_field_id is None:
    print("No numeric field found for EDA. Please manually set 'numeric_field_id' to the @id of a numeric column.")
else:
    threshold = chosen_df[numeric_field_id].mean() if pd.api.types.is_numeric_dtype(chosen_df[numeric_field_id]) else 10
    filtered_df = chosen_df[chosen_df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}")
    print(filtered_df.head())

    # Normalize
    filtered_df = filtered_df.copy()  # avoid pandas SettingWithCopyWarning
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields. Here, we show a histogram or boxplot for the numeric field, and a bar plot for grouped means by a categorical field if available.

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline

if numeric_field_id:
    plt.figure(figsize=(6,4))
    chosen_df[numeric_field_id].hist(bins=30)
    plt.xlabel(numeric_field_id)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.show()

    if group_field_id and group_field_id in chosen_df.columns:
        # Only plot if low cardinality
        n_groups = chosen_df[group_field_id].nunique()
        if n_groups < 20:
            grouped = chosen_df.groupby(group_field_id)[numeric_field_id].mean()
            grouped.plot(kind='bar', figsize=(8,4))
            plt.ylabel(f"Mean {numeric_field_id}")
            plt.title(f"Mean {numeric_field_id} by {group_field_id}")
            plt.show()

## 6. Conclusion
We demonstrated how to load, inspect, filter, and visualize the [FAIR² dataset package](https://sen.science/doi/10.71728/senscience.vq4a-28xa) using `mlcroissant`, referencing all data entities by their `@id`. You can now proceed to more advanced analyses or adapt these techniques for downstream tasks such as machine learning, annotation curation, or FAIR compliance evaluation.